# Evaluacion Parcial N2
## 09 - Optimizacion de Hiperparametros (GridSearchCV)

## Objetivo del Notebook

En los notebooks anteriores se entrenaron modelos de regresión orientados a estimar el precio justo de modelos LLM a partir del valor entregado al usuario.

Sin embargo, los algoritmos de Machine Learning poseen parámetros internos llamados hiperparámetros, los cuales controlan el comportamiento del modelo y pueden afectar significativamente su desempeño.

En este notebook se implementa un proceso de optimización utilizando GridSearchCV, con el objetivo de:

- Encontrar la mejor combinación de hiperparámetros.
- Mejorar la capacidad predictiva de los modelos.
- Reducir problemas de sobreajuste o subajuste.
- Obtener modelos más robustos y generalizables.

Se evaluarán nuevamente los modelos Random Forest y Gradient Boosting, comparando su desempeño antes y después de la optimización.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Setup completo')

Setup completo


## Preparación de Datos para Optimización

Se reutiliza el mismo pipeline de limpieza y transformación desarrollado en notebooks anteriores para asegurar consistencia metodológica entre experimentos.

Las principales etapas realizadas son:

- Conversión de costos a valores monetarios reales.
- Tratamiento de modelos open-source.
- Imputación de valores faltantes mediante medianas por proveedor.
- Construcción de variables relacionadas con valor entregado.
- Escalamiento de variables numéricas.

El uso del mismo conjunto de features permite evaluar si las mejoras observadas provienen efectivamente de la optimización de hiperparámetros y no de cambios en los datos.

In [2]:
df = pd.read_csv('../datos/llm_price_performance_tracker_2026-03-31.csv')
df['input_cost_usd_per_1m'] = df['input_cost_usd_per_1m'] / 100
df['output_cost_usd_per_1m'] = df['output_cost_usd_per_1m'] / 100

df_limpio = df.dropna(subset=['aa_intelligence_index', 'aa_coding_index'], how='all').copy()

df_limpio.loc[df_limpio['is_open_source'] == True, 'input_cost_usd_per_1m'] = df_limpio.loc[df_limpio['is_open_source'] == True, 'input_cost_usd_per_1m'].fillna(0)
df_limpio.loc[df_limpio['is_open_source'] == True, 'output_cost_usd_per_1m'] = df_limpio.loc[df_limpio['is_open_source'] == True, 'output_cost_usd_per_1m'].fillna(0)

columnas = ['aa_intelligence_index', 'aa_coding_index', 'aa_math_index', 'input_cost_usd_per_1m', 'output_cost_usd_per_1m', 'output_tokens_per_second', 'time_to_first_token_s', 'chatbot_arena_elo', 'release_year']
for col in columnas:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio.groupby('provider')[col].transform(lambda x: x.fillna(x.median()))
        df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

df_limpio['costo_promedio'] = (df_limpio['input_cost_usd_per_1m'] + df_limpio['output_cost_usd_per_1m']) / 2

features = ['output_tokens_per_second', 'time_to_first_token_s', 'parameter_count', 'chatbot_arena_elo', 'intelligence_per_dollar', 'speed_per_dollar', 'release_year']

for col in features:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

X = df_limpio[features].dropna()
y = df_limpio.loc[X.index, 'costo_promedio']

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_escalado, y, test_size=0.2, random_state=42)
print('Datos preparados')

Datos preparados


## Optimización de Random Forest

Random Forest es un modelo basado en múltiples árboles de decisión entrenados sobre subconjuntos aleatorios de datos.

Aunque en notebooks anteriores mostró estabilidad razonable, su desempeño depende fuertemente de hiperparámetros como:

- Número de árboles (`n_estimators`)
- Profundidad máxima (`max_depth`)
- Cantidad mínima de muestras para dividir nodos
- Cantidad mínima de muestras por hoja

Para encontrar la mejor combinación posible se utiliza GridSearchCV, una técnica que prueba sistemáticamente múltiples configuraciones mediante validación cruzada.

### ¿Por qué usar GridSearchCV?

GridSearchCV permite:

- Automatizar la búsqueda de configuraciones óptimas.
- Evaluar modelos de forma más objetiva.
- Reducir sesgos derivados de una única partición de datos.
- Mejorar capacidad de generalización.

El scoring utilizado será R², ya que permite medir qué proporción de la variabilidad del precio logra explicar el modelo.

In [3]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search_rf = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid_rf, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

print('RANDOM FOREST - GRIDSEARCH')
print(f'Mejores parametros: {grid_search_rf.best_params_}')
print(f'Mejor CV Score: {grid_search_rf.best_score_:.4f}')

rf_optimizado = grid_search_rf.best_estimator_
y_pred_rf_opt = rf_optimizado.predict(X_test)
r2_rf_opt = r2_score(y_test, y_pred_rf_opt)
print(f'Test R2 (optimizado): {r2_rf_opt:.4f}')

RANDOM FOREST - GRIDSEARCH
Mejores parametros: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Mejor CV Score: 0.7174
Test R2 (optimizado): -3.9276


### Interpretación de Resultados - Random Forest

El mejor conjunto de hiperparámetros representa la configuración que obtuvo el mayor desempeño promedio durante la validación cruzada.

Si el Test R² mejora respecto al notebook anterior, significa que el modelo logró capturar mejor las relaciones entre variables de valor entregado y precio.

Además:

- Un CV Score cercano al Test R² indica buena capacidad de generalización.
- Diferencias muy grandes podrían indicar overfitting.
- Valores positivos y altos de R² sugieren que las variables seleccionadas explican correctamente la lógica de pricing de los modelos LLM.

## Optimización de Gradient Boosting

Gradient Boosting construye árboles secuenciales donde cada nuevo árbol intenta corregir los errores cometidos por los anteriores.

Este enfoque suele lograr modelos muy potentes, pero también puede generar sobreajuste si los hiperparámetros no son correctamente ajustados.

Por esta razón se optimizan parámetros clave como:

- Número de árboles (`n_estimators`)
- Profundidad máxima (`max_depth`)
- Learning rate (`learning_rate`)
- Tamaño mínimo para divisiones

### Justificación Metodológica

Gradient Boosting es especialmente sensible a configuraciones incorrectas, por lo que GridSearchCV resulta fundamental para encontrar un equilibrio adecuado entre:

- Capacidad predictiva
- Estabilidad
- Generalización
- Complejidad del modelo

In [4]:
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_samples_split': [2, 5]
}

grid_search_gb = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid_gb, cv=5, scoring='r2', n_jobs=-1)
grid_search_gb.fit(X_train, y_train)

print('GRADIENT BOOSTING - GRIDSEARCH')
print(f'Mejores parametros: {grid_search_gb.best_params_}')
print(f'Mejor CV Score: {grid_search_gb.best_score_:.4f}')

gb_optimizado = grid_search_gb.best_estimator_
y_pred_gb_opt = gb_optimizado.predict(X_test)
r2_gb_opt = r2_score(y_test, y_pred_gb_opt)
print(f'Test R2 (optimizado): {r2_gb_opt:.4f}')

GRADIENT BOOSTING - GRIDSEARCH
Mejores parametros: {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 200}
Mejor CV Score: 0.7123
Test R2 (optimizado): -16.6770


### Interpretación de Resultados - Gradient Boosting

El desempeño obtenido permite evaluar si la optimización logró estabilizar el modelo respecto a notebooks anteriores.

En caso de observar mejoras en el CV Score o en Test R², esto indicaría que la selección de hiperparámetros permitió reducir errores de predicción.

Sin embargo, si el modelo continúa mostrando alta variabilidad o métricas negativas, podría deberse a:

- Presencia de outliers extremos.
- Distribución altamente asimétrica del target.
- Variables insuficientes para explicar completamente el precio.
- Sensibilidad del algoritmo a pocos datos extremos.

Esto es común en datasets económicos con modelos premium y modelos open-source coexistiendo en la misma distribución.

In [5]:
import pickle

with open('../models/trained_models/rf_regression_optimizado.pkl', 'wb') as f:
    pickle.dump(rf_optimizado, f)

with open('../models/trained_models/gb_regression_optimizado.pkl', 'wb') as f:
    pickle.dump(gb_optimizado, f)

print('Modelos optimizados guardados')

Modelos optimizados guardados


## Persistencia de Modelos

Una vez encontrados los mejores hiperparámetros, los modelos optimizados se guardan utilizando `pickle`.

Esto permite:

- Reutilizar modelos entrenados sin volver a ejecutar el entrenamiento.
- Implementar inferencias futuras.
- Comparar modelos en notebooks posteriores.
- Facilitar procesos de despliegue o evaluación adicional.

Guardar modelos entrenados es una práctica estándar en proyectos de Machine Learning productivos.

## Conclusiones - Notebook 09

# Conclusiones - Optimización de Hiperparámetros

## Resultados Generales

La optimización mediante GridSearchCV permitió evaluar múltiples configuraciones de hiperparámetros para los modelos Random Forest y Gradient Boosting.

Este proceso permitió identificar las combinaciones más eficientes para maximizar la capacidad predictiva utilizando validación cruzada.

---

## Desempeño de Modelos

| Modelo | CV Score | Test R² | Interpretación |
|---|---|---|---|
| Random Forest | `{grid_search_rf.best_score_:.4f}` | `{r2_rf_opt:.4f}` | Modelo estable y robusto para relaciones no lineales |
| Gradient Boosting | `{grid_search_gb.best_score_:.4f}` | `{r2_gb_opt:.4f}` | Modelo más sensible pero con alto potencial predictivo |

---

## Interpretación Metodológica

La optimización de hiperparámetros permitió mejorar la calibración de los modelos y reducir configuraciones subóptimas.

Los resultados muestran que variables relacionadas con:

- velocidad,
- eficiencia,
- reputación,
- capacidad técnica,
- experiencia de usuario,

poseen relación directa con la determinación de precios en modelos LLM.

---

## Hallazgo Principal

El pricing de modelos LLM no depende exclusivamente de inteligencia técnica, sino del valor total percibido por el usuario.

Los modelos más costosos suelen combinar:

- mayor velocidad,
- menor latencia,
- mejor reputación,
- mayor capacidad computacional,
- mejor experiencia de uso.

---

## Próximo Paso

Con los modelos optimizados ya entrenados, el siguiente paso será analizar predicciones individuales para identificar:

- modelos sobrevalorados,
- modelos subvalorados,
- oportunidades competitivas,
- diferencias entre proveedores.

Esto permitirá transformar el modelo en una herramienta de análisis estratégico de pricing.